# 해석적 X형 새들 주기의 Monodromy 행렬과 궤도 이동

이 notebook은 3-D toroidal geometry 위의 **해석적으로 구성한 saddle cycle**, 즉 Poincaré map의 X-point에서 pyna Monodromy workflow의 핵심을 보여 줍니다.

**학습 내용:**
1. 원통좌표 $(R,Z,φ)$에서 알려진 m=3/n=1 X-cycle을 갖는 해석 field를 정의하는 방법
2. `evolve_DPm_along_cycle`이 변분방정식 `dDX_pol/dφ = A · DX_pol`을 적분해 Monodromy 행렬 `DPm`을 얻는 방법
3. DPm evolution, 즉 commutator equation에 왜 identity가 아니라 `DPm(φ₀)=DX_pol(φ_end)`가 필요한지와 그 기하학적 의미
4. `orbit_shift_under_perturbation`으로 perturbation field 아래 **linear orbit shift**를 계산하는 방법
5. 해석 공식을 pyna API 결과와 비교하는 방법

**Reference geometry:**
Major radius R₀=1 m, $B_φ^{ax}=2.5$ T를 갖는 model tokamak, semi-axes $(R_{ell}=0.3, Z_{ell}=0.5)$인 elliptical X-cycle, rotational transform $ι=n/m=1/3$.

## 1. 해석적 field 구성

**정확한** m=3 X-cycle이 해석적으로 알려진 field를 구성합니다.

### 1.1 Cycle과 고유방향

Cycle은 다음 ellipse 위에 놓입니다.
$$
  X_c(\varphi)=\begin{pmatrix} R_{\rm ax}+R_{\rm ell}\cos(\iota\varphi+\varphi_0) \\
                                  Z_{\rm ax}+Z_{\rm ell}\sin(\iota\varphi+\varphi_0) \end{pmatrix}
$$
$\iota=1/3$이므로 orbit은 $m=3$ toroidal turns 뒤 닫힙니다.

X-point에서 Poincaré map의 고유값은 $\lambda_u=e^{+1/5}$, $\lambda_s=e^{-1/5}$입니다. 고유방향은 미리 정한 angle $\theta(\varphi)$를 따라 orbit 위에서 회전합니다.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# ── Cycle parameters ────────────────────────────────────────────────────────
Rax, Zax = 1.0, 0.0
Rell, Zell = 0.3, 0.5
phi0_cycle = 0.0
BPhiax = 2.5
m = 3       # toroidal period (orbit closes after m turns)
n = 1
iota = n / m  # rotational transform

# Eigenvalues of the m-turn Poincaré map
lam_u = np.e ** (1/5)   # unstable multiplier
lam_s = np.e ** (-1/5)  # stable  multiplier
print(f"λ_u = {lam_u:.6f},  λ_s = {lam_s:.6f},  product = {lam_u * lam_s:.10f}  (should be 1)")

# Rotating eigendirections along the orbit
def theta_u(phi):
    """Angle of unstable eigenvector at toroidal position phi."""
    return phi/3 + np.pi/2 + np.pi/9

def theta_s(phi):
    """Angle of stable eigenvector at toroidal position phi."""
    return phi/3 + np.pi/2 - np.pi/9

dtheta_dphi = 1.0/3  # same rotation rate as the orbit

# ── The cycle position ──────────────────────────────────────────────────────
def cycle_RZ(phi):
    """Return (R, Z) of the X-cycle at toroidal angle phi."""
    return np.array([
        Rell * np.cos(iota * phi + phi0_cycle) + Rax,
        Zell * np.sin(iota * phi + phi0_cycle) + Zax,
    ])

# ── Toroidal field (axisymmetric model: R·B_φ = const) ─────────────────────
def BPhi(phi, RZ):
    """Toroidal field B_φ(R) = B_φ^ax · R_ax / R  (free-force-balance model)."""
    return BPhiax * Rax / RZ[0]

print("Cycle at phi=0:", cycle_RZ(0.0))
print("Cycle at phi=2π:", cycle_RZ(2*np.pi))
print("Cycle at phi=6π (closes):", cycle_RZ(6*np.pi))

λ_u = 1.221403,  λ_s = 0.818731,  product = 1.0000000000  (should be 1)
Cycle at phi=0: [1.3 0. ]
Cycle at phi=2π: [0.85      0.4330127]
Cycle at phi=6π (closes): [ 1.3000000e+00 -1.2246468e-16]


In [2]:
# ── Poloidal field on the cycle ──────────────────────────────────────────────
#
# The poloidal field on the exact cycle is just B_pol = (dR_c/dl, dZ_c/dl)
# re-parameterised by phi.  We compute the φ-derivatives:
#   dR_c/dφ = -ι R_ell sin(ι φ + φ₀)
#   dZ_c/dφ = +ι Z_ell cos(ι φ + φ₀)

def BR0_BZ0_on_cycle(phi):
    """Return (B_R, B_Z) of the unperturbed field on the X-cycle."""
    Rc = cycle_RZ(phi)[0]
    Bp = BPhi(phi, cycle_RZ(phi))
    dRc = -iota * Rell * np.sin(iota * phi + phi0_cycle)
    dZc =  iota * Zell * np.cos(iota * phi + phi0_cycle)
    return np.array([dRc / Rc * Bp, dZc / Rc * Bp])


# ── Full field B(R, Z, φ) with exact X-cycle ────────────────────────────────
#
# We construct B_pol(X, φ) so that:
#  1. B_pol(X_c(φ), φ) = B_pol^0(φ)  (matches cycle tangent)
#  2. The Jacobian A(φ) of g = R·B_pol/B_φ along X_c has the prescribed
#     eigenstructure (λ_u, λ_s with rotating eigenvectors θ_u, θ_s).
#
# Construction: let V(φ) = [e_u | e_s] be the eigenvector matrix and
# Λ = diag(log|λ_u|, log|λ_s|) / (2mπ)  (per-φ growth rates).
# Then  A(φ) = V Λ V^{-1} + dV/dφ · V^{-1}  (accounts for rotating frame).
# The field is B_pol(X, φ) = B_pol^0(φ) + R·A(φ)·(X - X_c(φ)) / R_c

def _eigvec_matrix(phi):
    """2x2 matrix of eigenvectors [e_u | e_s] at phi."""
    return np.array([
        [np.cos(theta_u(phi)), np.cos(theta_s(phi))],
        [np.sin(theta_u(phi)), np.sin(theta_s(phi))],
    ])

def _A_matrix(phi):
    """A(φ) = Jacobian of g = R·B_pol/B_φ w.r.t. (R,Z) along cycle."""
    V = _eigvec_matrix(phi)
    # per-radian growth rates
    mu_u = np.log(np.abs(lam_u)) / (2 * m * np.pi)
    mu_s = np.log(np.abs(lam_s)) / (2 * m * np.pi)
    Lam = np.diag([mu_u, mu_s])
    # rotating-frame correction: dV/dφ · V^{-1}
    dV = np.array([
        [-dtheta_dphi * np.sin(theta_u(phi)), -dtheta_dphi * np.sin(theta_s(phi))],
        [ dtheta_dphi * np.cos(theta_u(phi)),  dtheta_dphi * np.cos(theta_s(phi))],
    ])
    return V @ Lam @ np.linalg.inv(V) + dV @ np.linalg.inv(V)

def field_g_pol(phi, X_pol):
    """φ-parameterised field direction g = R·B_pol/B_φ  (units: m)."""
    Xc = cycle_RZ(phi)
    g0 = Xc  # Xc gives the cycle tangent when dividing by R·B_phi/R_B_phi^ax
    # Actually g = dX_c/dphi + A·(X - X_c)
    dXc = np.array([
        -iota * Rell * np.sin(iota * phi + phi0_cycle),
         iota * Zell * np.cos(iota * phi + phi0_cycle),
    ])
    A = _A_matrix(phi)
    return dXc + A @ (X_pol - Xc)

# Verify: g on the cycle equals dX_c/dphi
phi_test = 0.7
Xc_test = cycle_RZ(phi_test)
g_on_cycle = field_g_pol(phi_test, Xc_test)
dXc_expected = np.array([
    -iota * Rell * np.sin(iota * phi_test + phi0_cycle),
     iota * Zell * np.cos(iota * phi_test + phi0_cycle),
])
print("g on cycle  :", g_on_cycle)
print("dX_c/dφ     :", dXc_expected)
print("Match:", np.allclose(g_on_cycle, dXc_expected))

g on cycle  : [-0.02312218  0.16215018]
dX_c/dφ     : [-0.02312218  0.16215018]
Match: True


## 2. `pyna` field function으로 wrapping

pyna는 signature가 `field_func(rzphi) -> (dR/dl, dZ/dl, dφ/dl)`인 field function, 즉 arc-length parameterisation을 기대합니다. $d\varphi/dl=B_\varphi/(R|B|)$를 사용해 φ-parameterisation에서 변환합니다.

In [3]:
def pyna_field(rzphi):
    """pyna-compatible field function: rzphi=(R,Z,φ) ?(dR/dl, dZ/dl, dφ/dl)."""
    R, Z, phi = rzphi[0], rzphi[1], rzphi[2]
    X_pol = np.array([R, Z])
    g = field_g_pol(phi, X_pol)          # (dR/dφ, dZ/dφ)
    Bp = BPhi(phi, X_pol)                # B_φ at (R, Z)
    # dφ/dl = B_φ / (R · |B|),  |B|² = B_R² + B_Z² + B_φ²
    # B_R = g[0] · B_φ / R,  B_Z = g[1] · B_φ / R
    # So  |B|² = B_φ² · (g²/R² + 1)  ? dphi/dl = 1/sqrt(R² + |g|²)
    dphi_dl = 1.0 / np.sqrt(R**2 + np.dot(g, g))
    dR_dl = g[0] * dphi_dl
    dZ_dl = g[1] * dphi_dl
    return np.array([dR_dl, dZ_dl, dphi_dl])

# Quick sanity: integrate the cycle once and check it closes
def rhs_phi(phi, XZ):
    f = pyna_field(np.array([XZ[0], XZ[1], phi]))
    dphi_dl = f[2]
    return np.array([f[0] / dphi_dl, f[1] / dphi_dl])  # dX/dphi

X0 = cycle_RZ(0.0)
sol = solve_ivp(rhs_phi, [0, 2 * m * np.pi], X0, rtol=1e-10, atol=1e-12, max_step=0.01)
X_end = sol.y[:, -1]
print(f"Cycle start: {X0}")
print(f"Cycle end  : {X_end}")
print(f"Closure error: {np.linalg.norm(X_end - X0):.2e}  (should be < 1e-6)")

Cycle start: [1.3 0. ]
Cycle end  : [1.30000000e+00 3.20923843e-17]
Closure error: 4.45e-16  (should be < 1e-6)


## 3. `pyna.topo.evolve_DPm_along_cycle`로 Monodromy 행렬 계산

`evolve_DPm_along_cycle` 함수는 orbit을 따라 두 개의 coupled equation을 적분합니다.

**Phase 1** -- `DX_pol`에 대한 변분방정식:
$$
  \frac{d\,\mathtt{DX\_pol}}{d\varphi}=A(\varphi)\cdot\mathtt{DX\_pol},
  \quad \mathtt{DX\_pol}(\varphi_0)=I
$$
한 period 전체($\varphi_0\to\varphi_0+2m\pi$)를 지난 뒤,
$\mathtt{DPm}=\mathtt{DX\_pol}(\varphi_{\rm end})$를 얻습니다. 이것이 Monodromy 행렬입니다.

**Phase 2** -- DPm evolution에 대한 commutator equation:
$$
  \frac{d\,\mathtt{DPm}}{d\varphi}=A\,\mathtt{DPm}-\mathtt{DPm}\,A,
  \quad \mathtt{DPm}(\varphi_0)=\mathtt{DX\_pol}(\varphi_{\rm end})
$$
초기조건은 **identity가 아닙니다**. Phase 1에서 방금 계산한 Monodromy 행렬입니다. 이렇게 해야 `DPm(φ)`가 period를 `φ₀` 대신 `φ`에서 시작했을 때 full-period map Jacobian이 어떤 모습인지 추적합니다.

In [4]:
from pyna.topo.monodromy import evolve_DPm_along_cycle
from pyna.topo.toroidal_cycle import ToroidalPeriodicOrbitTrace as PeriodicOrbit

# Build a PeriodicOrbit object that pyna needs
rzphi0 = np.array([cycle_RZ(0.0)[0], cycle_RZ(0.0)[1], 0.0])
orbit = PeriodicOrbit(
    rzphi0=rzphi0,
    period_m=m,
    trajectory=np.zeros((1, 3)),  # placeholder; evolve_DPm_along_cycle re-integrates
    DPm=np.eye(2),                # placeholder
)

# Run monodromy computation
mono = evolve_DPm_along_cycle(
    field_func=pyna_field,
    orbit=orbit,
    dt_output=2 * np.pi / 200,  # ~200 output points per turn
    rtol=1e-10, atol=1e-12,
)

print("DPm (monodromy matrix):")
print(mono.DPm)
print()
print("Eigenvalues :", mono.eigenvalues)
print("Stability index (Tr/2) :", mono.stability_index)
print("Greene residue         :", mono.Greene_residue)

DPm (monodromy matrix):
[[ 1.02006674 -0.07328026]
 [-0.55316615  1.02006674]]

Eigenvalues : [0.81873081 1.22140267]
Stability index (Tr/2) : 1.0200667408183737
Greene residue         : -0.010033370409186837


In [5]:
# ── Analytic verification ────────────────────────────────────────────────────
#
# The analytic DPm at phi=0 should be V(2mπ) · diag(λ_u, λ_s) · V^{-1}(0)
# where V(φ) is the eigenvector matrix.

V_end = _eigvec_matrix(2 * m * np.pi)
V_start = _eigvec_matrix(0.0)
DPm_analytic = V_end @ np.diag([lam_u, lam_s]) @ np.linalg.inv(V_start)

print("DPm analytic:")
print(DPm_analytic)
print()
print("DPm from pyna:")
print(mono.DPm)
print()
print("Max error:", np.max(np.abs(mono.DPm - DPm_analytic)))

DPm analytic:
[[ 1.02006676 -0.07328031]
 [-0.55316612  1.02006676]]

DPm from pyna:
[[ 1.02006674 -0.07328026]
 [-0.55316615  1.02006674]]

Max error: 5.4677180783002655e-08


## 4. Orbit 위의 DX_pol과 DPm 초기조건

여기서는 각 φ에서 state transition matrix인 `DX_pol(φ)`를 시각화하고, Phase-2 DPm equation이 왜 `I`가 아니라 `DX_pol(φ_end)`에서 시작하는지 설명합니다.

**Geometric meaning:** `DPm(φ)`는 φ₀ 대신 φ에서 시작하는 "shifted" orbit의 Monodromy입니다. Poincaré section을 옮기면 full-period map도 바뀌므로 일반적으로 `DPm(φ)`는 `DX_pol(φ_end)`와 같지 않습니다. 정의와 일관되려면 commutator equation으로 진화시키되 초기값은 `DX_pol(φ_end)`와 같아야 합니다.

In [6]:
phi_arr = mono.phi_arr
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
fig.suptitle("DX_pol(φ) components along the m=3 X-cycle", fontsize=14)
labels = [["DX_pol[0,0]", "DX_pol[0,1]"], ["DX_pol[1,0]", "DX_pol[1,1]"]]

for i in range(2):
    for j in range(2):
        axes[i, j].plot(phi_arr / np.pi, mono.DX_pol_arr[:, i, j], color='steelblue')
        axes[i, j].set_ylabel(labels[i][j])
        axes[i, j].axvline(0, color='k', lw=0.5)
        axes[i, j].axvline(2 * m, color='gray', lw=0.5, linestyle='--', label='φ_end')
        # Mark the initial value of DPm (= DX_pol at phi_end)
        axes[i, j].scatter([2 * m], [mono.DX_pol_arr[-1, i, j]],
                            color='red', zorder=5, label='DPm IC')

axes[0, 0].legend(fontsize=9)
for ax in axes[1]:
    ax.set_xlabel("φ / π")
plt.tight_layout()
plt.savefig("DX_pol_components.png", dpi=120)
plt.show()

print("DX_pol(φ_end) = DPm_init (Phase-2 IC):")
print(mono.DX_pol_arr[-1])
print()
print("DPm (Phase-1 result = monodromy):")
print(mono.DPm)
print()
print("They are the same:", np.allclose(mono.DX_pol_arr[-1], mono.DPm))

DX_pol(φ_end) = DPm_init (Phase-2 IC):
[[ 1.02006674 -0.07328026]
 [-0.55316615  1.02006674]]

DPm (Phase-1 result = monodromy):
[[ 1.02006674 -0.07328026]
 [-0.55316615  1.02006674]]

They are the same: True


## 5. Perturbation field 아래 linear orbit shift

이제 **small perturbation** `δB`를 적용하고 `orbit_shift_under_perturbation`으로 X-cycle의 linear shift를 계산합니다.

Shift `δX(φ)`는 다음을 만족합니다.
$$
  \frac{d\,\delta X}{d\varphi}=A(\varphi)\,\delta X+\delta b_{\rm pol}(\varphi)
$$
**periodic boundary condition** `δX(φ₀ + 2mπ) = δX(φ₀)`를 적용하면,
$$
  \delta X(\varphi_0)=(\mathtt{DPm}-I)^{-1}\,(-\delta X_{\rm particular}(\varphi_{\rm end}))
$$
를 얻습니다.

여기서는 단순한 해석 perturbation인 uniform vertical shift `δB_Z = ε`를 사용합니다.

In [7]:
from pyna.topo.monodromy import orbit_shift_under_perturbation

epsilon = 0.01  # perturbation amplitude

def delta_field(rzphi):
    """Perturbation: uniform δB_Z = ε (arc-length parameterised)."""
    R, Z, phi = rzphi[0], rzphi[1], rzphi[2]
    X_pol = np.array([R, Z])
    Bp = BPhi(phi, X_pol)
    # dφ/dl from unperturbed field
    g = field_g_pol(phi, X_pol)
    dphi_dl = 1.0 / np.sqrt(R**2 + np.dot(g, g))
    # δB_Z = ε ?δ(dZ/dl) = ε · |B| / |B| = ε / |B| · |B| = ε dphi_dl · R / Bp · Bp = ...
    # Simple: δdZ/dl = ε · dphi_dl / dphi_dl = ε (unit B)
    return np.array([0.0, epsilon * dphi_dl, 0.0])

delta_X = orbit_shift_under_perturbation(
    field_func=pyna_field,
    delta_field_func=delta_field,
    orbit=orbit,
    monodromy_analysis=mono,
)

print(f"Orbit shift at φ=0: δR={delta_X[0, 0]:.6f} m,  δZ={delta_X[0, 1]:.6f} m")
print(f"Periodicity check: |δX(end) - δX(start)| = "
      f"{np.linalg.norm(delta_X[-1] - delta_X[0]):.2e}  (should be < 1e-6)")

Orbit shift at φ=0: δR=-0.029622 m,  δZ=0.000000 m
Periodicity check: |δX(end) - δX(start)| = 1.36e-15  (should be < 1e-6)


In [8]:
# ── Visualise the orbit shift in 3D ─────────────────────────────────────────
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

Rc = mono.trajectory[:, 0]
Zc = mono.trajectory[:, 1]
phi_plot = mono.phi_arr

# Unperturbed cycle (xyz)
ax.plot(Rc * np.cos(phi_plot), Rc * np.sin(phi_plot), Zc,
        color='black', linewidth=2, label='Unperturbed X-cycle')

# Shifted cycle
R_shifted = Rc + delta_X[:, 0]
Z_shifted = Zc + delta_X[:, 1]
ax.plot(R_shifted * np.cos(phi_plot), R_shifted * np.sin(phi_plot), Z_shifted,
        color='tomato', linewidth=2, linestyle='--', label=f'Shifted cycle (ε={epsilon})')

ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_zlabel('Z [m]')
ax.legend()
ax.set_title('Linear orbit shift under uniform δB_Z perturbation')
plt.tight_layout()
plt.savefig("orbit_shift_3d.png", dpi=120)
plt.show()

## 6. Key takeaways

| Concept | Symbol | pyna API |
|---------|--------|----------|
| Orbit 위 variational matrix | `DX_pol(φ)` | `mono.DX_pol_arr`, `mono.DX_pol_at(φ)` |
| Full-period Poincaré map Jacobian | `DPm` | `mono.DPm` |
| Phase-2 initial condition | `DPm(φ₀)=DX_pol(φ_end)` | `evolve_DPm_along_cycle`에서 자동 처리 |
| DPm eigenvalues | `λ_u, λ_s` | `mono.eigenvalues` |
| Stability index | `Tr(DPm)/2` | `mono.stability_index` |
| Greene residue | `(2 - Tr(DPm))/4` | `mono.Greene_residue` |
| Linear orbit shift | `δX(φ)` | `orbit_shift_under_perturbation(...)` |

**Why `DX_pol` not `J`?**
Subscript `_pol`은 이 matrix가 Cartesian이 아니라 cylindrical 또는 polar $(R,Z)$ space에 산다는 점을 분명히 합니다. 변수명이 물리적 의미를 바로 전달합니다.

**Why `DPm` not `M`?**
`DPm`은 *D*erivative of *P*oincaré *m*ap, 즉 m-turn period의 derivative입니다. 단일 문자 `M`은 이 풍부한 구조를 드러내지 못합니다.